
### CREATE FLAG PARAMETER

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text('incremental_flag', '0')

#It is in string type
#So we have to use '0' or '1' when we are using it 

In [0]:
incremental_flag = dbutils.widgets.get('incremental_flag')


### CREATING DIMENSION MODEL

In [0]:
%sql

SELECT * FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/car_sales`


### Gold - Dim_Branch

In [0]:
df_src = spark.sql('''
                     SELECT * FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/car_sales`
                     ''')
display(df_src)

In [0]:
df_branch = spark.sql('''
                     SELECT DISTINCT(Branch_ID) AS Branch_ID, BranchName FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/car_sales`
                     ''')
display(df_branch)

In [0]:
#If Table not exists - just bring the schema

if spark.catalog.tableExists('cars_catalog.gold.dim_branch'):
    df_branch_sink = spark.sql('''
    SELECT dim_branch_key, Branch_ID , BranchName FROM cars_cata.gold.dim_branch
    ''')
else:
    df_branch_sink = spark.sql('''
    SELECT 1 as dim_branch_key, Branch_ID , BranchName FROM parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/car_sales` WHERE 1=0
    ''')
 

In [0]:
display(df_branch_sink)

In [0]:
# Filtering new records and old records

df_filter = df_branch.join(df_branch_sink, df_branch.Branch_ID == df_branch_sink.Branch_ID, 'left').select(df_branch['Branch_ID'], df_branch['BranchName'], df_branch_sink['dim_branch_key'])




In [0]:
df_filter_old = df_filter.filter(col("dim_branch_key").isNotNull())
df_filter_new = df_filter.filter(col("dim_branch_key").isNull()).select("Branch_ID", "BranchName")

In [0]:
display(df_filter_new)


#### Create Surrogate Key


###### Fetch the Max Surrogate Key 

In [0]:
#Initial run - start from 1
#Incremental run - start from max(key)+1 

if incremental_flag == '0':
    max_value = 0
else:
    max_value_df = spark.sql(''' SELECT max(dim_branch_key) FROM   cars_cata.gold.dim_model ''')
    max_value = max_value_df.collect()[0][0]


###### Create Surrogate Key

In [0]:
df_filter_new = df_filter_new.withColumn('dim_branch_key', max_value+monotonically_increasing_id()+1)

In [0]:
display(df_filter_new)



#### Create final df

In [0]:
df_final = df_filter_new.union(df_filter_old)
display(df_final)


### SCD TYPE 1 - UPSERT

In [0]:
#if - incremental run else for initial run

if spark.catalog.tableExists('cars_cata.gold.dim_branch'):
    delta_tbl = DeltaTable.forPath(spark,'abfss://gold@adlscarproject.dfs.core.windows.net/dim_branch' )
    delta_tbl.alias("trg").merge(df_final.alias("src"),"trg.dim_branch_key = src.dim_branch_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_final.write.format("delta")\
        .mode('overwrite')\
        .option('path', 'abfss://gold@adlscarproject.dfs.core.windows.net/dim_branch')\
        .saveAsTable('cars_cata.gold.dim_branch')

In [0]:
%sql
SELECT * FROM cars_cata.gold.dim_branch